# Proyek Portofolio Data Analis: SF Restaurant Inspection Scores
## Bagian 1: Pembersihan Data (Data Cleaning) & Penataan Data (*Data Preprocessing*)

**Penulis:** Gilang Anugrah  
**Dataset:** San Francisco Restaurant Inspection Scores (LIVES Standard)  
**Tujuan Notebook:**  
Mengubah data mentah (*raw data*) menjadi dataset bersih (*processed data*) yang siap dianalisis dan divisualisasikan. 
Setiap tahap pembersihan data didasarkan pada audit menyeluruh terhadap karakteristik dan logika domain inspeksi sanitasi restoran.

---
### 📌 Rangkuman Tahapan Pembersihan:
1. **Audit Kualitas Data Awal** (tipe data, missing values, duplikat identik).
2. **Penanganan Baris Duplikat Identik** (menghapus duplikasi baris persis).
3. **Standardisasi Tanggal & Ekstraksi Fitur Waktu** (tahun, bulan, tahun-bulan, hari, kuartal).
4. **Pembersihan Anomali Kode Pos** (mengambil 5 digit awal, validasi format pos).
5. **Koreksi Koordinat Geografis Dummy** (mengubah koordinat `(0, 0)` menjadi `NaN`).
6. **Pembersihan Teks & Format Kontak** (trim spasi, perbaikan format nomor telepon).
7. **Penanganan Kontekstual Pelanggaran & Kategorisasi Skor** (membedakan inspeksi bersih vs pelanggaran).
8. **Pembuatan Tabel Agregasi Tingkat Inspeksi** (mengatasi bias duplikasi per pelanggaran).
9. **Ekspor Dataset Bersih** ke folder `data/processed/`.

In [1]:
# 1. Mengimpor pustaka utama untuk manipulasi data, sistem file, dan visualisasi tampilan
import os
import pandas as pd
import numpy as np
from IPython.display import display

# Mengatur opsi tampilan pandas agar seluruh kolom terlihat jelas saat inspeksi
pd.set_option('display.max_columns', None)
print("Library berhasil dimuat.")

Library berhasil dimuat.


---
### Tahap 1: Memuat Data Mentah (*Raw Data*)
Membaca dataset asli dari folder `data/raw/` tanpa mengubah file aslinya.

In [2]:
# Menentukan lokasi file data mentah
raw_file_path = os.path.join('..', 'data', 'raw', 'Restaurant_Scores_-_LIVES_Standard.csv')
if not os.path.exists(raw_file_path):
    raw_file_path = os.path.join('data', 'raw', 'Restaurant_Scores_-_LIVES_Standard.csv')

# Membaca data mentah ke dalam DataFrame pandas
df_raw = pd.read_csv(raw_file_path)

# Menampilkan dimensi data mentah dan 3 baris pertama
print(f"Dimensi Data Mentah: {df_raw.shape[0]:,} baris x {df_raw.shape[1]} kolom")
display(df_raw.head(3))

Dimensi Data Mentah: 53,555 baris x 17 kolom


,business_id,business_name,business_address,business_city,business_state,business_postal_code,business_latitude,business_longitude,business_location,business_phone_number,inspection_id,inspection_date,inspection_score,inspection_type,violation_id,violation_description,risk_category
0,10,Tiramisu Kitchen,033 Belden Pl,San Francisco,CA,94104,37.791116,-122.403816,"(37.791116, -122.403816)",NaN,10_20140114,01/14/2014 12:00:00 AM,92.0,Routine - Unscheduled,10_20140114_103119,Inadequate and inaccessible handwashing facili...,Moderate Risk
1,10,Tiramisu Kitchen,033 Belden Pl,San Francisco,CA,94104,37.791116,-122.403816,"(37.791116, -122.403816)",NaN,10_20140114,01/14/2014 12:00:00 AM,92.0,Routine - Unscheduled,10_20140114_103145,Improper storage of equipment utensils or linens,Low Risk
2,10,Tiramisu Kitchen,033 Belden Pl,San Francisco,CA,94104,37.791116,-122.403816,"(37.791116, -122.403816)",NaN,10_20140114,01/14/2014 12:00:00 AM,92.0,Routine - Unscheduled,10_20140114_103154,Unclean or degraded floors walls or ceilings,Low Risk


---
### Tahap 2: Audit Kualitas Data Awal (*Data Quality Auditing*)
Memeriksa tipe data, jumlah nilai kosong (*missing values*), serta keberadaan baris duplikat.

In [3]:
# Memeriksa tipe data dan ringkasan nilai kosong pada setiap kolom
data_audit = pd.DataFrame({
    'Tipe Data': df_raw.dtypes,
    'Jumlah Missing': df_raw.isnull().sum(),
    'Persentase Missing (%)': round((df_raw.isnull().sum() / len(df_raw)) * 100, 2)
})

# Menampilkan tabel audit kualitas data
print("=== RINGKASAN AUDIT KUALITAS DATA AWAL ===")
display(data_audit)

# Menghitung jumlah baris duplikat identik di seluruh kolom
total_duplikat = df_raw.duplicated().sum()
print(f"Jumlah baris duplikat identik: {total_duplikat} baris")

=== RINGKASAN AUDIT KUALITAS DATA AWAL ===


,Tipe Data,Jumlah Missing,Persentase Missing (%)
business_id,int64,0,0.00
business_name,str,0,0.00
business_address,str,0,0.00
business_city,str,0,0.00
business_state,str,0,0.00
business_postal_code,str,1357,2.53
business_latitude,float64,17632,32.92
business_longitude,float64,17632,32.92
business_location,str,17632,32.92
business_phone_number,float64,37747,70.48


Jumlah baris duplikat identik: 50 baris


---
### Tahap 3: Menghapus Baris Duplikat Identik
Baris yang memiliki nilai identik di semua kolom merupakan redundansi input data dan harus dihapus.

In [4]:
# Membuat salinan data kerja (working copy) dan menghapus baris duplikat
df_clean = df_raw.drop_duplicates().copy()

# Memverifikasi jumlah baris setelah duplikat dihapus
print(f"Baris sebelum: {len(df_raw):,} | Baris sesudah: {len(df_clean):,}")
print(f"Berhasil menghapus: {len(df_raw) - len(df_clean)} baris duplikat.")

Baris sebelum: 53,555 | Baris sesudah: 53,505
Berhasil menghapus: 50 baris duplikat.


---
### Tahap 4: Standardisasi Tipe Tanggal & Ekstraksi Fitur Waktu (*Time Features*)
Kolom `inspection_date` awalnya bertipe string/object. Kita konversi ke `datetime` dan mengekstrak tahun, bulan, tahun-bulan, hari dalam seminggu, dan kuartal untuk mempermudah analisis tren waktu.

In [5]:
# Mengonversi inspection_date dari string ke tipe datetime
df_clean['inspection_date'] = pd.to_datetime(df_clean['inspection_date'], format='%m/%d/%Y %I:%M:%S %p')

# Mengekstrak komponen waktu sebagai fitur baru untuk analisis tren
df_clean['inspection_year'] = df_clean['inspection_date'].dt.year
df_clean['inspection_month'] = df_clean['inspection_date'].dt.month
df_clean['inspection_year_month'] = df_clean['inspection_date'].dt.to_period('M').astype(str)
df_clean['inspection_day_name'] = df_clean['inspection_date'].dt.day_name()
df_clean['inspection_quarter'] = 'Q' + df_clean['inspection_date'].dt.quarter.astype(str)

# Memeriksa rentang tanggal data yang berhasil dikonversi
print(f"Tanggal Inspeksi Paling Awal : {df_clean['inspection_date'].min().date()}")
print(f"Tanggal Inspeksi Paling Akhir: {df_clean['inspection_date'].max().date()}")
print()
print("Distribusi Data per Tahun:")
display(df_clean['inspection_year'].value_counts().sort_index())

Tanggal Inspeksi Paling Awal : 2014-01-02
Tanggal Inspeksi Paling Akhir: 2016-12-30

Distribusi Data per Tahun:


inspection_year
2014    18457
2015    13425
2016    21623
Name: count, dtype: int64

---
### Tahap 5: Pembersihan Anomali Kolom Kode Pos (`business_postal_code`)
Pemeriksaan menemukan beberapa anomali pada kode pos:
- Format 9 digit tanpa tanda hubung (misal: `941102019` -> diambil 5 digit pertama `94110`).
- Kesalahan input singkatan negara bagian (`Ca`, `CA`) dan kode terpotong (`941`) -> diubah menjadi `NaN` agar tidak mengacaukan analisis wilayah.

In [6]:
# Fungsi khusus untuk memvalidasi dan membersihkan format kode pos San Francisco
def bersihkan_kode_pos(val):
    if pd.isnull(val):
        return np.nan
    val_str = str(val).strip()
    # Mengabaikan input singkatan state ('Ca'/'CA') atau kode yang terpotong ('941')
    if val_str.upper() == 'CA' or val_str == '941':
        return np.nan
    # Mengambil 5 digit pertama jika merupakan format angka 5 atau 9 digit
    if len(val_str) >= 5 and val_str[:5].isdigit():
        return val_str[:5]
    return np.nan

# Menerapkan fungsi pembersihan pada kolom business_postal_code
df_clean['business_postal_code'] = df_clean['business_postal_code'].apply(bersihkan_kode_pos)

# Menampilkan 5 kode pos terbanyak setelah dibersihkan
print("Top 5 Kode Pos dengan Jumlah Restoran/Inspeksi Terbanyak:")
display(df_clean['business_postal_code'].value_counts().head(5))
print(f"Jumlah nilai missing pada kode pos: {df_clean['business_postal_code'].isnull().sum():,} baris")

Top 5 Kode Pos dengan Jumlah Restoran/Inspeksi Terbanyak:


business_postal_code
94110    5063
94133    4995
94103    4316
94109    4183
94102    3658
Name: count, dtype: int64

Jumlah nilai missing pada kode pos: 1,394 baris


---
### Tahap 6: Koreksi Koordinat Geografis Dummy `(0.0, 0.0)`
Sebanyak 69 baris data (terutama gerai makanan keliling/food truck) memiliki koordinat `(0.0, 0.0)`. Koordinat ini berada di laut lepas Afrika ("Null Island") dan akan merusak peta sebaran jika tidak diubah menjadi `NaN`.

In [7]:
# Mendeteksi baris dengan koordinat dummy 0.0 pada latitude atau longitude
dummy_coords = (df_clean['business_latitude'] == 0) | (df_clean['business_longitude'] == 0)
print(f"Jumlah baris dengan koordinat (0, 0): {dummy_coords.sum()} baris")

# Mengubah nilai koordinat dummy menjadi NaN
df_clean.loc[dummy_coords, ['business_latitude', 'business_longitude', 'business_location']] = np.nan

# Memverifikasi nilai minimum koordinat setelah dikoreksi (harus di area San Francisco)
print(f"Min Latitude  : {df_clean['business_latitude'].min():.4f} (Area SF)")
print(f"Min Longitude : {df_clean['business_longitude'].min():.4f} (Area SF)")

Jumlah baris dengan koordinat (0, 0): 69 baris
Min Latitude  : 37.6688 (Area SF)
Min Longitude : -122.5109 (Area SF)


---
### Tahap 7: Standardisasi Format Teks & Nomor Kontak Telepon
Menghapus spasi berlebih pada kolom teks dan mengubah format nomor telepon dari float notasi ilmiah (`1.41558e+10`) menjadi string angka yang rapi.

In [8]:
# Menghapus spasi di awal dan akhir pada kolom berbasis teks
kolom_teks = ['business_name', 'business_address', 'business_city', 'business_state']
for col in kolom_teks:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype(str).str.strip()

# Fungsi untuk memformat nomor telepon dari float ilmiah ke string angka bersih
def format_telepon(phone):
    if pd.isnull(phone):
        return np.nan
    try:
        return str(int(phone))
    except (ValueError, TypeError):
        return np.nan

# Menerapkan pembersihan format nomor telepon
df_clean['business_phone_number'] = df_clean['business_phone_number'].apply(format_telepon)

print("Contoh nomor telepon setelah diformat:")
display(df_clean['business_phone_number'].dropna().head(3))

Contoh nomor telepon setelah diformat:


5     14155829718
8     14155816139
17    14155548833
Name: business_phone_number, dtype: str

---
### Tahap 8: Penanganan Logika Domain (Pelanggaran & Kategori Skor)
Berdasarkan aturan bisnis SFDPH:
1. `violation_id` yang bernilai `NaN` menandakan **inspeksi bersih tanpa pelanggaran**. Kita tandai dengan kolom boolean `is_clean_inspection`.
2. Missing value pada `risk_category` diisi dengan label `'No Violation'`.
3. Kolom `inspection_score` dikelompokkan ke dalam kategori standar mutu sanitasi:
   - **Good / Excellent**: 90 – 100
   - **Adequate**: 80 – 89
   - **Needs Improvement**: 70 – 79
   - **Poor**: < 70
   - **Unscored / Non-Routine**: Inspeksi tanpa skor numerik (kunjungan ulang, keluhan, dll.)

In [9]:
# Menandai inspeksi yang bersih tanpa pelanggaran
df_clean['is_clean_inspection'] = df_clean['violation_id'].isnull()

# Memberi label eksplisit pada kategori risiko jika tidak ada pelanggaran
df_clean['risk_category_clean'] = df_clean['risk_category'].fillna('No Violation')

# Fungsi untuk mengelompokkan skor numerik ke dalam tingkatan mutu standar SFDPH
def beri_kategori_skor(skor):
    if pd.isnull(skor):
        return 'Unscored / Non-Routine'
    elif skor >= 90:
        return 'Good / Excellent (90-100)'
    elif skor >= 80:
        return 'Adequate (80-89)'
    elif skor >= 70:
        return 'Needs Improvement (70-79)'
    else:
        return 'Poor (<70)'

# Menerapkan kategorisasi skor
df_clean['score_category'] = df_clean['inspection_score'].apply(beri_kategori_skor)

# Menampilkan distribusi kategori skor
print("Distribusi Kategori Skor Inspeksi (Tingkat Baris Pelanggaran):")
display(df_clean['score_category'].value_counts())

Distribusi Kategori Skor Inspeksi (Tingkat Baris Pelanggaran):


score_category
Good / Excellent (90-100)    17763
Adequate (80-89)             14294
Unscored / Non-Routine       12761
Needs Improvement (70-79)     6345
Poor (<70)                    2342
Name: count, dtype: int64

---
### Tahap 9: Pembuatan Tabel Agregasi Tingkat Inspeksi (*Inspection-level Grain*)
> **Penting untuk Best Practice Data Analis:**
> Dataset utama memiliki granularitas per pelanggaran. Jika kita menghitung rata-rata skor langsung dari dataset utama, inspeksi dengan banyak pelanggaran akan terhitung berkali-kali (*double-counting bias*).
> 
> Oleh karena itu, kita membuat tabel agregasi di mana **1 baris = 1 sesi inspeksi unik**, lengkap dengan ringkasan jumlah pelanggaran High, Moderate, dan Low Risk.

In [10]:
# Mengagregasi data berdasarkan inspection_id unik
df_inspeksi = df_clean.groupby('inspection_id').agg(
    business_id=('business_id', 'first'),
    business_name=('business_name', 'first'),
    business_address=('business_address', 'first'),
    business_postal_code=('business_postal_code', 'first'),
    business_latitude=('business_latitude', 'first'),
    business_longitude=('business_longitude', 'first'),
    inspection_date=('inspection_date', 'first'),
    inspection_year=('inspection_year', 'first'),
    inspection_month=('inspection_month', 'first'),
    inspection_year_month=('inspection_year_month', 'first'),
    inspection_day_name=('inspection_day_name', 'first'),
    inspection_quarter=('inspection_quarter', 'first'),
    inspection_type=('inspection_type', 'first'),
    inspection_score=('inspection_score', 'first'),
    score_category=('score_category', 'first'),
    is_clean_inspection=('is_clean_inspection', 'first'),
    total_violations=('violation_id', lambda x: x.notnull().sum()),
    high_risk_count=('risk_category', lambda x: (x == 'High Risk').sum()),
    moderate_risk_count=('risk_category', lambda x: (x == 'Moderate Risk').sum()),
    low_risk_count=('risk_category', lambda x: (x == 'Low Risk').sum())
).reset_index()

# Membuat kolom indikator apakah ada pelanggaran High Risk dalam inspeksi tersebut
df_inspeksi['has_high_risk'] = df_inspeksi['high_risk_count'] > 0

# Mengurutkan tabel agar rapi berdasarkan nomor business_id dan tanggal inspeksi
df_inspeksi = df_inspeksi.sort_values(by=['business_id', 'inspection_date']).reset_index(drop=True)

# Menampilkan ringkasan tabel tingkat inspeksi
print(f"Dimensi Tabel Tingkat Inspeksi: {df_inspeksi.shape[0]:,} baris x {df_inspeksi.shape[1]} kolom")
print(f"Total sesi inspeksi bebas pelanggaran : {(df_inspeksi['total_violations'] == 0).sum():,} sesi")
print(f"Total sesi inspeksi dengan High Risk   : {df_inspeksi['has_high_risk'].sum():,} sesi")
display(df_inspeksi.head(3))

Dimensi Tabel Tingkat Inspeksi: 25,834 baris x 22 kolom
Total sesi inspeksi bebas pelanggaran : 12,882 sesi
Total sesi inspeksi dengan High Risk   : 4,123 sesi


,inspection_id,business_id,business_name,business_address,business_postal_code,business_latitude,business_longitude,inspection_date,inspection_year,inspection_month,inspection_year_month,inspection_day_name,inspection_quarter,inspection_type,inspection_score,score_category,is_clean_inspection,total_violations,high_risk_count,moderate_risk_count,low_risk_count,has_high_risk
0,10_20140114,10,Tiramisu Kitchen,033 Belden Pl,94104,37.791116,-122.403816,2014-01-14,2014,1,2014-01,Tuesday,Q1,Routine - Unscheduled,92.0,Good / Excellent (90-100),False,3,0,1,2,False
1,10_20140124,10,Tiramisu Kitchen,033 Belden Pl,94104,37.791116,-122.403816,2014-01-24,2014,1,2014-01,Friday,Q1,Reinspection/Followup,NaN,Unscored / Non-Routine,True,0,0,0,0,False
2,10_20140729,10,Tiramisu Kitchen,033 Belden Pl,94104,37.791116,-122.403816,2014-07-29,2014,7,2014-07,Tuesday,Q3,Routine - Unscheduled,94.0,Good / Excellent (90-100),False,2,0,1,1,False


---
### Tahap 10: Mengekspor Dataset Bersih ke Folder `data/processed/`
Menyimpan kedua hasil dataset bersih ke file CSV agar siap digunakan untuk analisis eksplorasi (EDA) dan pembuatan dashboard.

In [11]:
# Menentukan path folder output
output_dir = os.path.join('..', 'data', 'processed')
if not os.path.exists(output_dir):
    output_dir = os.path.join('data', 'processed')
os.makedirs(output_dir, exist_ok=True)

# Menentukan nama file hasil ekspor
file_clean_path = os.path.join(output_dir, 'sf_restaurant_inspections_cleaned.csv')
file_summary_path = os.path.join(output_dir, 'sf_inspections_summary.csv')

# Menyimpan kedua dataframe ke file CSV
df_clean.to_csv(file_clean_path, index=False)
df_inspeksi.to_csv(file_summary_path, index=False)

print("BERHASIL MENYIMPAN DATASET BERSIH:")
print(f"1. Detail Pelanggaran : {file_clean_path} ({df_clean.shape[0]:,} baris)")
print(f"2. Ringkasan Inspeksi : {file_summary_path} ({df_inspeksi.shape[0]:,} baris)")

BERHASIL MENYIMPAN DATASET BERSIH:
1. Detail Pelanggaran : ..\data\processed\sf_restaurant_inspections_cleaned.csv (53,505 baris)
2. Ringkasan Inspeksi : ..\data\processed\sf_inspections_summary.csv (25,834 baris)


---
### Tahap 11: Validasi Akhir & Ringkasan Perubahan (*Sanity Check*)

In [12]:
# Membuat tabel ringkasan perbandingan sebelum dan sesudah data cleaning
ringkasan_perubahan = pd.DataFrame({
    'Parameter': [
        'Total Baris Data',
        'Jumlah Duplikat Identik',
        'Rentang Tahun',
        'Jumlah Restoran Unik (business_id)',
        'Jumlah Inspeksi Unik (inspection_id)',
        'Kolom Baru Ditambahkan'
    ],
    'Sebelum Cleaning (Raw)': [
        f"{len(df_raw):,}",
        f"{df_raw.duplicated().sum():,}",
        "String text (MM/DD/YYYY)",
        f"{df_raw['business_id'].nunique():,}",
        f"{df_raw['inspection_id'].nunique():,}",
        "0"
    ],
    'Sesudah Cleaning (Processed)': [
        f"{len(df_clean):,}",
        "0",
        f"{df_clean['inspection_year'].min()} - {df_clean['inspection_year'].max()}",
        f"{df_clean['business_id'].nunique():,}",
        f"{df_clean['inspection_id'].nunique():,}",
        "8 kolom baru (fitur waktu, kategori, indikator)"
    ]
})

print("=== VALIDASI AKHIR PERBANDINGAN DATA ===")
display(ringkasan_perubahan)

=== VALIDASI AKHIR PERBANDINGAN DATA ===


,Parameter,Sebelum Cleaning (Raw),Sesudah Cleaning (Processed)
0,Total Baris Data,"53,555","53,505"
1,Jumlah Duplikat Identik,50,0
2,Rentang Tahun,String text (MM/DD/YYYY),2014 - 2016
3,Jumlah Restoran Unik (business_id),"6,078","6,078"
4,Jumlah Inspeksi Unik (inspection_id),"25,834","25,834"
5,Kolom Baru Ditambahkan,0,"8 kolom baru (fitur waktu, kategori, indikator)"


---
### Tahap 12: Download & Akses Langsung File Dataset Bersih
Sel di bawah ini memungkinkan Anda untuk langsung mengunduh dan mengakses file CSV yang sudah bersih:
1. **Tautan Unduh Interaktif (*FileLink*):** Bisa diklik langsung dari output sel untuk mengunduh via browser.
2. **Kompatibilitas Google Colab:** Jika notebook dijalankan di Colab, proses download file akan terpicu otomatis.
3. **Akses File Explorer Lokal (Windows):** Otomatis membuka folder `data/processed/` di File Explorer komputer Anda.

In [13]:
# Tahap 12: Menyediakan Tautan Download Interaktif dan Membuka Lokasi File Bersih
import os
from IPython.display import display, FileLink, HTML

# Mendapatkan lokasi absolut file dan folder hasil pembersihan
output_folder = os.path.abspath(output_dir)
clean_csv_abs = os.path.abspath(file_clean_path)
summary_csv_abs = os.path.abspath(file_summary_path)

print("=== STATUS FILE BERSIH SIAP DIUNDUH ===")
print(f"Lokasi Folder : {output_folder}")
print(f"File 1 (Detail Pelanggaran) : {clean_csv_abs}")
print(f"File 2 (Ringkasan Sesi)     : {summary_csv_abs}")
print()
print("Klik tautan di bawah ini untuk mengunduh:")

# 1. Menyediakan tautan unduh FileLink yang dapat langsung diklik pada antarmuka Jupyter
display(FileLink(file_clean_path, result_html_prefix="Unduh File 1: "))
display(FileLink(file_summary_path, result_html_prefix="Unduh File 2: "))

# 2. Mendukung pengunduhan otomatis jika pengguna menjalankan notebook ini di Google Colab
try:
    from google.colab import files
    print()
    print("Terdeteksi lingkungan Google Colab: Memulai download otomatis...")
    files.download(file_clean_path)
    files.download(file_summary_path)
except ImportError:
    pass

# 3. Membuka folder output secara otomatis di File Explorer Windows lokal
try:
    os.startfile(output_folder)
    print()
    print(f"Folder berhasil dibuka di File Explorer: {output_folder}")
except Exception:
    pass

=== STATUS FILE BERSIH SIAP DIUNDUH ===
Lokasi Folder : c:\Users\Gilang Anugrah\OneDrive\Dokumen\Gilang Anugrah\Portofolio-Data Analis\Portofolio 1\data\processed
File 1 (Detail Pelanggaran) : c:\Users\Gilang Anugrah\OneDrive\Dokumen\Gilang Anugrah\Portofolio-Data Analis\Portofolio 1\data\processed\sf_restaurant_inspections_cleaned.csv
File 2 (Ringkasan Sesi)     : c:\Users\Gilang Anugrah\OneDrive\Dokumen\Gilang Anugrah\Portofolio-Data Analis\Portofolio 1\data\processed\sf_inspections_summary.csv

Klik tautan di bawah ini untuk mengunduh:


c:\Users\Gilang Anugrah\OneDrive\Dokumen\Gilang Anugrah\Portofolio-Data Analis\Portofolio 1\data\processed\sf_restaurant_inspections_cleaned.csv

c:\Users\Gilang Anugrah\OneDrive\Dokumen\Gilang Anugrah\Portofolio-Data Analis\Portofolio 1\data\processed\sf_inspections_summary.csv


Folder berhasil dibuka di File Explorer: c:\Users\Gilang Anugrah\OneDrive\Dokumen\Gilang Anugrah\Portofolio-Data Analis\Portofolio 1\data\processed
